In [649]:
# import modules
import numpy as np
import tensorflow as tf
from tensorflow import keras
import pandas as pd
import seaborn as sns
from pylab import rcParams
import matplotlib.pyplot as plt
from sklearn.model_selection import GroupShuffleSplit
from scipy import stats
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split
from tensorflow.keras import optimizers
from datetime import datetime
from kerastuner.tuners import RandomSearch
import tensorflow_addons as tfa
from keras.utils import to_categorical 
from keras.regularizers import l2
import os



r_seed = 42

WORKDIR = 'C:/Users/thewr/git/LSTM_giroscopio_acelerometro.git/'
np.random.seed(r_seed)
tf.random.set_seed(r_seed)
data_hora= datetime.now().strftime('%Y-%m-%d_%H-%M-%S')
LOG_DIR = WORKDIR + "/Code/DataPrep/"+data_hora

In [650]:
date = datetime.now()
date = date.strftime("%d/%m/%Y_%H:%M")
arq = datetime.now()
arq = arq.strftime("%d_%m_%Y_%H_%M")

In [651]:
def combine_csv_data(directory):
    # Specify the directory containing the TXT files

    # Create an empty list to store the DataFrames
    dfs = []

    # Iterate through the TXT files in the directory
    for filename in os.listdir(directory):
        if filename.endswith(".txt"):
            # Read the CSV data from the TXT file, assuming no header row
            df = pd.read_csv(os.path.join(directory, filename), header=None)

            # Append the DataFrame to the list
            dfs.append(df)

    # Concatenate the DataFrames into a single DataFrame
    combined_df = pd.concat(dfs, ignore_index=True)

    # Print the combined DataFrame
    return combined_df

In [652]:
def equalize_dataframe_sizes(df1, df2):
    """Equalizes the sizes of two DataFrames by dropping rows from the larger one.

    Args:
        df1 (pd.DataFrame): The first DataFrame.
        df2 (pd.DataFrame): The second DataFrame.

    Returns:
        tuple: A tuple containing the equalized DataFrames (df1, df2).

    Raises:
        ValueError: If both DataFrames have the same size.
    """

    if df1.shape[0] == df2.shape[0]:
        raise ValueError("Both DataFrames already have the same size.")

    larger_df, smaller_df = (df1, df2) if df1.shape[0] > df2.shape[0] else (df2, df1)

    # Drop rows from the larger DataFrame to match the smaller one
    larger_df = larger_df.iloc[: smaller_df.shape[0], :]

    return larger_df, smaller_df

# Example usage
#df1 = pd.DataFrame({'A': [1, 2, 3, 4, 5], 'B': ['a', 'b', 'c', 'd', 'e']})
#df2 = pd.DataFrame({'A': [10, 20, 30], 'B': ['x', 'y', 'z']})

''' try:
    df1_equal, df2_equal = equalize_dataframe_sizes(df1.copy(), df2.copy())
    #print("Equalized DataFrames:")
    #print(df1_equal)
    #print(df2_equal)
except ValueError as e:
    print(e) '''


' try:\n    df1_equal, df2_equal = equalize_dataframe_sizes(df1.copy(), df2.copy())\n    #print("Equalized DataFrames:")\n    #print(df1_equal)\n    #print(df2_equal)\nexcept ValueError as e:\n    print(e) '

In [653]:
def custom_concat(df1, df2):
    """
    Concatenates two DataFrames with the same column names, considering specific conditions.

    Args:
        df1 (pd.DataFrame): The first DataFrame.
        df2 (pd.DataFrame): The second DataFrame.

    Returns:
        pd.DataFrame: The concatenated DataFrame.
    """

    result_df = pd.DataFrame()

    # Create groups based on unique combinations of X and Y values
    groups_df1 = df1.groupby(['pessoa_gyro','atividade_gyro'])
    groups_df2 = df2.groupby(['pessoa', 'atividade'])

    # Iterate over groups from both DataFrames simultaneously
    for (x, y), group1 in groups_df1:
        #print(group1)
        #if (x,y) in [(1637, 'C'),(1638, 'H'),(1638, 'O'),(1640, 'C'),(1640, 'E')]:
        #    continue
        #else:
        group2 = groups_df2.get_group((x, y))
        '''  group1_has_nan = group1.isnull().sum().sum() > 0
        group2_has_nan = group2.isnull().sum().sum() > 0
        if group1_has_nan == True:
                print(group1.isnull().sum().sum())
                print (group1)
                break
        if group2_has_nan == True:
                print(group2.isnull().sum().sum())
                print (group2)
                break '''
        
        if group2 is not None:  # Check if corresponding group exists in df2
            if group1.shape[0] != group2.shape[0]:
                group1_equal, group2_equal = equalize_dataframe_sizes(group1.copy(), group2.copy())
                #print(f"+++++++++++++++++++++++++++++++++Imprimindo shape grupo 1 e 2 :{group1_equal.shape[0]} - {group2_equal.shape[0]}" )
                #print(f"$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$Imprimindo dados grupo 1 :{group1_equal}" )
                #print("---------------------------------------------")
                #print(f"####################################Imprimindo dados grupo 2 :{group2_equal}" )
            #print(f"+++++++++++++++++++++++++++++++++Imprimindo shape grupo 1 e 2 :{group1_equal.shape[0]} - {group2_equal.shape[0]}" )
            #print(f"$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$Imprimindo dados grupo 1 :{group1_equal}" )
            #print("---------------------------------------------")
            #print(f"####################################Imprimindo dados grupo 2 :{group2_equal}" ) 
            group1_equal = group1_equal.reset_index(drop=True) 
            group2_equal = group2_equal.reset_index(drop=True)  
            df_concat_group1_group2 = pd.concat([group1_equal, group2_equal], axis=1)
            #print(df_concat_group1_group2)
            df_concat_group1_group2 = df_concat_group1_group2.dropna()            
            #print(df_concat_group1_group2)
            has_nan = df_concat_group1_group2.isnull().sum().sum() > 0
            if has_nan == True:
                print (f"**************************** Imprimindo concat co nan: {df_concat_group1_group2}")
                break
            df_concat_group1_group2 = df_concat_group1_group2.reset_index(drop=True) 
            result_df = result_df.reset_index(drop=True)
            result_df = pd.concat([result_df, df_concat_group1_group2])

    return result_df

In [654]:
''' def custom_concat(df1, df2):
    """
    Concatenates two DataFrames with the same column names, considering specific conditions.

    Args:
        df1 (pd.DataFrame): The first DataFrame.
        df2 (pd.DataFrame): The second DataFrame.

    Returns:
        pd.DataFrame: The concatenated DataFrame.
    """

    result_df = pd.DataFrame()

    # Iterate over consecutive groups in each DataFrame
    for x_val in df1['pessoa'].unique():
        group_df1 = df1[df1['pessoa'] == x_val]
        for y_val in group_df1['atividade'].unique():
            group1 = group_df1[group_df1['atividade'] == y_val]
            group2 = df2[(df2['pessoa'] == x_val) & (df2['atividade'] == y_val)]
            if not group2.empty:  # Check if corresponding group exists in df2
                result_df = pd.concat([result_df, pd.concat([group2,group1], keys=[1, 2], axis=1, ignore_index=True)])

    return result_df '''



' def custom_concat(df1, df2):\n    """\n    Concatenates two DataFrames with the same column names, considering specific conditions.\n\n    Args:\n        df1 (pd.DataFrame): The first DataFrame.\n        df2 (pd.DataFrame): The second DataFrame.\n\n    Returns:\n        pd.DataFrame: The concatenated DataFrame.\n    """\n\n    result_df = pd.DataFrame()\n\n    # Iterate over consecutive groups in each DataFrame\n    for x_val in df1[\'pessoa\'].unique():\n        group_df1 = df1[df1[\'pessoa\'] == x_val]\n        for y_val in group_df1[\'atividade\'].unique():\n            group1 = group_df1[group_df1[\'atividade\'] == y_val]\n            group2 = df2[(df2[\'pessoa\'] == x_val) & (df2[\'atividade\'] == y_val)]\n            if not group2.empty:  # Check if corresponding group exists in df2\n                result_df = pd.concat([result_df, pd.concat([group2,group1], keys=[1, 2], axis=1, ignore_index=True)])\n\n    return result_df '

In [655]:
''' # Sample DataFrames
df1 = pd.DataFrame({'X': [1600, 1600, 1600, 1600, 1600, 1601, 1601, 1601, 1601],
                   'Y': ['A', 'A', 'B', 'B', 'C', 'A', 'A', 'B', 'B']})
df2 = pd.DataFrame({'X': [1600, 1600, 1600, 1600, 1601, 1601, 1601, 1601, 1601],
                   'Y': ['A', 'A', 'B', 'B', 'A', 'B', 'B', 'B', 'B']})

def custom_concat(df1, df2):
    """
    Concatenates two DataFrames with the same column names, considering specific conditions.

    Args:
        df1 (pd.DataFrame): The first DataFrame.
        df2 (pd.DataFrame): The second DataFrame.

    Returns:
        pd.DataFrame: The concatenated DataFrame.
    """

    result_df = pd.DataFrame()

    # Iterate over consecutive groups in each DataFrame
    for x_val in df1['X'].unique():
        group_df1 = df1[df1['X'] == x_val]
        for y_val in group_df1['Y'].unique():
            group1 = group_df1[group_df1['Y'] == y_val]
            group2 = df2[(df2['X'] == x_val) & (df2['Y'] == y_val)]
            if not group2.empty:  # Check if corresponding group exists in df2
                result_df = pd.concat([result_df, pd.concat([group1, group2], keys=[1, 2], axis=1, ignore_index=False)])

    return result_df

# Concatenate the DataFrames
concatenated_df = custom_concat(df1.copy(), df2.copy()) 

print(concatenated_df) '''

' # Sample DataFrames\ndf1 = pd.DataFrame({\'X\': [1600, 1600, 1600, 1600, 1600, 1601, 1601, 1601, 1601],\n                   \'Y\': [\'A\', \'A\', \'B\', \'B\', \'C\', \'A\', \'A\', \'B\', \'B\']})\ndf2 = pd.DataFrame({\'X\': [1600, 1600, 1600, 1600, 1601, 1601, 1601, 1601, 1601],\n                   \'Y\': [\'A\', \'A\', \'B\', \'B\', \'A\', \'B\', \'B\', \'B\', \'B\']})\n\ndef custom_concat(df1, df2):\n    """\n    Concatenates two DataFrames with the same column names, considering specific conditions.\n\n    Args:\n        df1 (pd.DataFrame): The first DataFrame.\n        df2 (pd.DataFrame): The second DataFrame.\n\n    Returns:\n        pd.DataFrame: The concatenated DataFrame.\n    """\n\n    result_df = pd.DataFrame()\n\n    # Iterate over consecutive groups in each DataFrame\n    for x_val in df1[\'X\'].unique():\n        group_df1 = df1[df1[\'X\'] == x_val]\n        for y_val in group_df1[\'Y\'].unique():\n            group1 = group_df1[group_df1[\'Y\'] == y_val]\n            

In [656]:
## get dataset

# select only sensor data from smart watches
csv_gyro_dir = WORKDIR +"Data/Raw/watch/gyro"
csv_accel_dir = WORKDIR +"Data/Raw/watch/accel"

In [657]:
df_csv_gyro = combine_csv_data(csv_gyro_dir)

In [658]:
df_csv_gyro = df_csv_gyro.replace({';':''}, regex=True)

In [659]:
df_csv_gyro.columns = ['pessoa_gyro','atividade_gyro','tempo_gyro','giroscópio_x','giroscópio_Y','giroscópio_Z']

In [660]:
df_csv_gyro.head()

,pessoa_gyro,atividade_gyro,tempo_gyro,giroscópio_x,giroscópio_Y,giroscópio_Z
0,1600,A,90426757696641,0.314944,-1.022277,-0.3099616
1,1600,A,90426807196641,0.387382,-0.618541,-0.048971802
2,1600,A,90426856696641,0.070999,-0.209480,-0.1959783
3,1600,A,90426906196641,0.037975,0.254976,-0.1565635
4,1600,A,90426955696641,0.073129,0.719431,-0.0010349044


In [661]:
df_csv_gyro.tail()

,pessoa_gyro,atividade_gyro,tempo_gyro,giroscópio_x,giroscópio_Y,giroscópio_Z
3440337,1650,S,2426184371437252,-2.856065,2.057101,0.85393304
3440338,1650,S,2426184421424592,-2.911458,1.817417,1.2811041
3440339,1650,S,2426184471411932,-3.015854,1.550036,2.0576818
3440340,1650,S,2426184521399272,-3.218254,1.595842,3.0185504
3440341,1650,S,2426184571385262,-2.455525,0.866136,3.3541086


In [662]:
df_csv_gyro.shape

(3440342, 6)

In [663]:
df_csv_gyro.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3440342 entries, 0 to 3440341
Data columns (total 6 columns):
 #   Column          Dtype  
---  ------          -----  
 0   pessoa_gyro     int64  
 1   atividade_gyro  object 
 2   tempo_gyro      int64  
 3   giroscópio_x    float64
 4   giroscópio_Y    float64
 5   giroscópio_Z    object 
dtypes: float64(2), int64(2), object(2)
memory usage: 157.5+ MB


In [664]:
condition1 = (df_csv_gyro['pessoa_gyro']== 1637) & (df_csv_gyro['atividade_gyro']=='C')
condition2 = (df_csv_gyro['pessoa_gyro']== 1638) & (df_csv_gyro['atividade_gyro']=='O')
condition3 = (df_csv_gyro['pessoa_gyro']== 1640) & (df_csv_gyro['atividade_gyro']=='C')
condition4 = (df_csv_gyro['pessoa_gyro']== 1640) & (df_csv_gyro['atividade_gyro']=='E')
condition5 = (df_csv_gyro['pessoa_gyro']== 1638) & (df_csv_gyro['atividade_gyro']=='H')

In [665]:
df_csv_gyro_filtered = df_csv_gyro[~(condition1 | condition2 | condition3 | condition4 | condition5)]

In [666]:
df_csv_gyro_filtered.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3440342 entries, 0 to 3440341
Data columns (total 6 columns):
 #   Column          Dtype  
---  ------          -----  
 0   pessoa_gyro     int64  
 1   atividade_gyro  object 
 2   tempo_gyro      int64  
 3   giroscópio_x    float64
 4   giroscópio_Y    float64
 5   giroscópio_Z    object 
dtypes: float64(2), int64(2), object(2)
memory usage: 157.5+ MB


In [667]:
test = df_csv_gyro_filtered[(df_csv_gyro_filtered['pessoa_gyro']==1638) & (df_csv_gyro_filtered['atividade_gyro']=='H') ]

In [668]:
print(len(test))

0


In [669]:
test2 = df_csv_gyro_filtered[df_csv_gyro_filtered['pessoa_gyro']==1638]

In [670]:
print(len(test2))

71426


In [671]:
df_csv_gyro_filtered.shape

(3440342, 6)

In [672]:
#df_csv_gyro.set_index(["pessoa_gyro","atividade_gyro","tempo_gyro"],inplace=True)

In [673]:
#df_csv_gyro.head()

In [674]:
df_csv_gyro.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3440342 entries, 0 to 3440341
Data columns (total 6 columns):
 #   Column          Dtype  
---  ------          -----  
 0   pessoa_gyro     int64  
 1   atividade_gyro  object 
 2   tempo_gyro      int64  
 3   giroscópio_x    float64
 4   giroscópio_Y    float64
 5   giroscópio_Z    object 
dtypes: float64(2), int64(2), object(2)
memory usage: 157.5+ MB


In [675]:
df_csv_accel = combine_csv_data(csv_accel_dir)

In [676]:
df_csv_accel = df_csv_accel.replace({';':''}, regex=True)

In [677]:
df_csv_accel.shape

(3777046, 6)

In [678]:
df_csv_accel.columns = ['pessoa','atividade','tempo_acelerômetro','acelerômetro_x','acelerômetro_Y','acelerômetro_Z']

In [679]:
df_csv_accel.head()

,pessoa,atividade,tempo_acelerômetro,acelerômetro_x,acelerômetro_Y,acelerômetro_Z
0,1600,A,90426708196641,7.091625,-0.591667,8.195502
1,1600,A,90426757696641,4.972757,-0.158317,6.6967316
2,1600,A,90426807196641,3.253720,-0.191835,6.107758
3,1600,A,90426856696641,2.801216,-0.155922,5.997625
4,1600,A,90426906196641,3.770868,-1.051354,7.731027


In [680]:
df_csv_accel.tail()

,pessoa,atividade,tempo_acelerômetro,acelerômetro_x,acelerômetro_Y,acelerômetro_Z
3777041,1650,S,2426184421424592,1.486500,-13.302334,5.355231
3777042,1650,S,2426184471411932,4.017171,-13.146711,5.1182046
3777043,1650,S,2426184521399272,4.412214,-15.318253,3.3704374
3777044,1650,S,2426184571385262,4.426579,-13.012636,2.5755625
3777045,1650,S,2426184621371252,3.765780,-11.798776,2.166154


In [681]:
#df_csv_accel.set_index(["pessoa_acelerômetro","atividade_acelerômetro","tempo_acelerômetro"],inplace=True)

In [682]:
#df_csv_accel.head()

In [683]:
condition_accel1 = (df_csv_accel['pessoa']== 1637) & (df_csv_accel['atividade']=='C')
condition_accel2 = (df_csv_accel['pessoa']== 1638) & (df_csv_accel['atividade']=='O')
condition_accel3 = (df_csv_accel['pessoa']== 1640) & (df_csv_accel['atividade']=='C')
condition_accel4 = (df_csv_accel['pessoa']== 1640) & (df_csv_accel['atividade']=='E')
condition_accel5 = (df_csv_accel['pessoa']== 1638) & (df_csv_accel['atividade']=='H')

In [684]:
df_csv_accel_filtered = df_csv_accel[~(condition_accel1 | condition_accel2 | condition_accel3 | condition_accel4 | condition_accel5)]

In [685]:
df_csv_accel_filtered.info()

<class 'pandas.core.frame.DataFrame'>
Index: 3732382 entries, 0 to 3777045
Data columns (total 6 columns):
 #   Column              Dtype  
---  ------              -----  
 0   pessoa              int64  
 1   atividade           object 
 2   tempo_acelerômetro  int64  
 3   acelerômetro_x      float64
 4   acelerômetro_Y      float64
 5   acelerômetro_Z      object 
dtypes: float64(2), int64(2), object(2)
memory usage: 199.3+ MB


In [686]:
test3 = df_csv_accel_filtered[(df_csv_accel_filtered['pessoa']==1638) & (df_csv_accel_filtered['atividade']=='H') ]

In [687]:
print(len(test3))

0


In [688]:
test3.head()

,pessoa,atividade,tempo_acelerômetro,acelerômetro_x,acelerômetro_Y,acelerômetro_Z


In [689]:
test4 = df_csv_accel_filtered[df_csv_accel_filtered['pessoa']==1638 ]

In [690]:
print(len(test4))

142932


In [691]:
df_csv_gyro_filtered.shape

(3440342, 6)

In [692]:
# Select rows with matching indices (inner join)
''' common_indices = df_csv_accel.index.intersection(df_csv_gyro.index)
df1_filtered = df_csv_accel.loc[common_indices]
df2_filtered = df_csv_gyro.loc[common_indices]

# Concatenate horizontally
result = pd.concat([df1_filtered, df2_filtered], axis=1) '''

' common_indices = df_csv_accel.index.intersection(df_csv_gyro.index)\ndf1_filtered = df_csv_accel.loc[common_indices]\ndf2_filtered = df_csv_gyro.loc[common_indices]\n\n# Concatenate horizontally\nresult = pd.concat([df1_filtered, df2_filtered], axis=1) '

In [713]:
result_concatenation = custom_concat(df_csv_gyro_filtered.copy(),df_csv_accel_filtered.copy())

In [714]:
result_concatenation.shape

(3441622, 12)

In [715]:
result_concatenation.head()

,pessoa,atividade,tempo_acelerômetro,acelerômetro_x,acelerômetro_Y,acelerômetro_Z,pessoa_gyro,atividade_gyro,tempo_gyro,giroscópio_x,giroscópio_Y,giroscópio_Z
0,1600,A,90426708196641,7.091625,-0.591667,8.195502,1600,A,90426757696641,0.314944,-1.022277,-0.3099616
1,1600,A,90426757696641,4.972757,-0.158317,6.6967316,1600,A,90426807196641,0.387382,-0.618541,-0.048971802
2,1600,A,90426807196641,3.253720,-0.191835,6.107758,1600,A,90426856696641,0.070999,-0.209480,-0.1959783
3,1600,A,90426856696641,2.801216,-0.155922,5.997625,1600,A,90426906196641,0.037975,0.254976,-0.1565635
4,1600,A,90426906196641,3.770868,-1.051354,7.731027,1600,A,90426955696641,0.073129,0.719431,-0.0010349044


In [716]:
result_concatenation = result_concatenation.reset_index(drop=True)

In [717]:
result_concatenation.tail()


,pessoa,atividade,tempo_acelerômetro,acelerômetro_x,acelerômetro_Y,acelerômetro_Z,pessoa_gyro,atividade_gyro,tempo_gyro,giroscópio_x,giroscópio_Y,giroscópio_Z
3441617,1650,S,2426184321449912,4.768950,-5.401469,7.5842323,1650,S,2426184371437252,-2.856065,2.057101,0.85393304
3441618,1650,S,2426184371437252,2.137723,-7.211485,8.065467,1650,S,2426184421424592,-2.911458,1.817417,1.2811041
3441619,1650,S,2426184421424592,1.486500,-13.302334,5.355231,1650,S,2426184471411932,-3.015854,1.550036,2.0576818
3441620,1650,S,2426184471411932,4.017171,-13.146711,5.1182046,1650,S,2426184521399272,-3.218254,1.595842,3.0185504
3441621,1650,S,2426184521399272,4.412214,-15.318253,3.3704374,1650,S,2426184571385262,-2.455525,0.866136,3.3541086


In [718]:
test_diff_atividade = result_concatenation[result_concatenation['atividade']!=result_concatenation['atividade_gyro']]

In [719]:
print(test_diff_atividade.shape[0])

0


In [720]:
test_diff_pessoa = result_concatenation[result_concatenation['pessoa']!=result_concatenation['pessoa_gyro']]

In [721]:
print(len(test_diff_pessoa))

0


In [722]:
result_concatenation.columns

Index(['pessoa', 'atividade', 'tempo_acelerômetro', 'acelerômetro_x',
       'acelerômetro_Y', 'acelerômetro_Z', 'pessoa_gyro', 'atividade_gyro',
       'tempo_gyro', 'giroscópio_x', 'giroscópio_Y', 'giroscópio_Z'],
      dtype='object')

In [723]:
result_concatenation = result_concatenation[['pessoa', 'atividade', 'tempo_acelerômetro', 'acelerômetro_x',
       'acelerômetro_Y', 'acelerômetro_Z', 'tempo_gyro', 'giroscópio_x', 'giroscópio_Y', 'giroscópio_Z']]

In [724]:
result_concatenation.head()

,pessoa,atividade,tempo_acelerômetro,acelerômetro_x,acelerômetro_Y,acelerômetro_Z,tempo_gyro,giroscópio_x,giroscópio_Y,giroscópio_Z
0,1600,A,90426708196641,7.091625,-0.591667,8.195502,90426757696641,0.314944,-1.022277,-0.3099616
1,1600,A,90426757696641,4.972757,-0.158317,6.6967316,90426807196641,0.387382,-0.618541,-0.048971802
2,1600,A,90426807196641,3.253720,-0.191835,6.107758,90426856696641,0.070999,-0.209480,-0.1959783
3,1600,A,90426856696641,2.801216,-0.155922,5.997625,90426906196641,0.037975,0.254976,-0.1565635
4,1600,A,90426906196641,3.770868,-1.051354,7.731027,90426955696641,0.073129,0.719431,-0.0010349044


In [726]:
result_concatenation = result_concatenation.rename(columns={'pessoa':'pessoa_x'})

In [727]:
result_concatenation.head()

,pessoa_x,atividade,tempo_acelerômetro,acelerômetro_x,acelerômetro_Y,acelerômetro_Z,tempo_gyro,giroscópio_x,giroscópio_Y,giroscópio_Z
0,1600,A,90426708196641,7.091625,-0.591667,8.195502,90426757696641,0.314944,-1.022277,-0.3099616
1,1600,A,90426757696641,4.972757,-0.158317,6.6967316,90426807196641,0.387382,-0.618541,-0.048971802
2,1600,A,90426807196641,3.253720,-0.191835,6.107758,90426856696641,0.070999,-0.209480,-0.1959783
3,1600,A,90426856696641,2.801216,-0.155922,5.997625,90426906196641,0.037975,0.254976,-0.1565635
4,1600,A,90426906196641,3.770868,-1.051354,7.731027,90426955696641,0.073129,0.719431,-0.0010349044


In [728]:
# Define the mapping dictionary
mapping_dict = {'A':0,'B':1,'C':2,'D':3,'E':4,'F':5,'G':6,'H':7,'I':8,'J':9,'K':10,'L':11,'M':12,'O':13,'P':14,'Q':15,'R':16,'S':17}

In [729]:
result_concatenation["atividade"] = result_concatenation["atividade"].replace(to_replace=mapping_dict)

In [730]:
result_concatenation.head()

,pessoa_x,atividade,tempo_acelerômetro,acelerômetro_x,acelerômetro_Y,acelerômetro_Z,tempo_gyro,giroscópio_x,giroscópio_Y,giroscópio_Z
0,1600,0,90426708196641,7.091625,-0.591667,8.195502,90426757696641,0.314944,-1.022277,-0.3099616
1,1600,0,90426757696641,4.972757,-0.158317,6.6967316,90426807196641,0.387382,-0.618541,-0.048971802
2,1600,0,90426807196641,3.253720,-0.191835,6.107758,90426856696641,0.070999,-0.209480,-0.1959783
3,1600,0,90426856696641,2.801216,-0.155922,5.997625,90426906196641,0.037975,0.254976,-0.1565635
4,1600,0,90426906196641,3.770868,-1.051354,7.731027,90426955696641,0.073129,0.719431,-0.0010349044


In [731]:
result_concatenation.tail()

,pessoa_x,atividade,tempo_acelerômetro,acelerômetro_x,acelerômetro_Y,acelerômetro_Z,tempo_gyro,giroscópio_x,giroscópio_Y,giroscópio_Z
3441617,1650,17,2426184321449912,4.768950,-5.401469,7.5842323,2426184371437252,-2.856065,2.057101,0.85393304
3441618,1650,17,2426184371437252,2.137723,-7.211485,8.065467,2426184421424592,-2.911458,1.817417,1.2811041
3441619,1650,17,2426184421424592,1.486500,-13.302334,5.355231,2426184471411932,-3.015854,1.550036,2.0576818
3441620,1650,17,2426184471411932,4.017171,-13.146711,5.1182046,2426184521399272,-3.218254,1.595842,3.0185504
3441621,1650,17,2426184521399272,4.412214,-15.318253,3.3704374,2426184571385262,-2.455525,0.866136,3.3541086


In [732]:
result_concatenation.to_csv('activity_recognition_data_custom_concat.csv', index=False)

In [ ]:
import pandas as pd

# Sample DataFrames (adjust values as needed)
df1 = pd.DataFrame({'X': [1600, 1600, 1600, 1600, 1600, 1601, 1601, 1601, 1601],
                   'Y': ['A', 'A', 'B', 'B', 'C', 'A', 'A', 'B', 'B']})

df2 = pd.DataFrame({'X': [1600, 1600, 1600, 1600, 1601, 1601, 1601, 1601, 1601],
                   'Y': ['A', 'B', 'A', 'B', 'A', 'B', 'B', 'B', 'B']})

# Function to concatenate DataFrames with specific conditions
def conditional_concat(df1, df2):
    """
    Concatenates two DataFrames horizontally under specific conditions.

    Args:
        df1 (pd.DataFrame): The first DataFrame.
        df2 (pd.DataFrame): The second DataFrame.

    Returns:
        pd.DataFrame: The concatenated DataFrame.
    """

    # Create a temporary DataFrame to store matching rows
    temp_df = pd.DataFrame(columns=df1.columns)

    # Iterate through each row of df1
    for i in df1.index:
        # Check if the row in df1 satisfies the conditions
        if df2[(df2['X'] == df1.loc[i, 'X']) & (df2['Y'] == df1.loc[i, 'Y'])].empty:
            continue  # Skip if no match found in df2

        # Add the matching row from df2 to temp_df
        matching_row = df2[(df2['X'] == df1.loc[i, 'X']) & (df2['Y'] == df1.loc[i, 'Y'])].iloc[0]
        temp_df = pd.concat([temp_df, matching_row], ignore_index=True)

    # Concatenate df1 and temp_df horizontally
    result = pd.concat([df1, temp_df], axis=1)

    return result

# Concatenate the DataFrames
concatenated_df = conditional_concat(df1, df2)

print(concatenated_df)


In [ ]:
import pandas as pd


f1 = pd.DataFrame({'X': [1600, 1600, 1600, 1600, 1600, 1601, 1601, 1601, 1601],
                   'Y': ['A', 'A', 'B', 'B', 'C', 'A', 'A', 'B', 'B']})

df2 = pd.DataFrame({'X': [1600, 1600, 1600, 1600, 1601, 1601, 1601, 1601, 1601],
                   'Y': ['A', 'B', 'A', 'B', 'A', 'B', 'B', 'B', 'B']})


def g(df1, df2):
    return pd.concat([df1[df1['X'].isin(df2['X']) & df1['Y'].isin(df2['Y'])],
                     df2[df2['X'].isin(df1['X']) & df2['Y'].isin(df1['Y'])]], sort=False).reset_index(drop=True)

result = g(f1.copy(),df2.copy())
print(result)


In [ ]:

import pandas as pd


f1 = pd.DataFrame({'X': [1600, 1600, 1600, 1600, 1600, 1601, 1601, 1601, 1601],
                   'Y': ['A', 'A', 'B', 'B', 'C', 'A', 'A', 'B', 'B']})

df2 = pd.DataFrame({'X': [1600, 1600, 1600, 1600, 1601, 1601, 1601, 1601, 1601],
                   'Y': ['A', 'B', 'A', 'B', 'A', 'B', 'B', 'B', 'B']})


def g(df1, df2):
  return pd.merge(df1, df2.loc[(df2['X'].isin(df1['X'])) & (df2['Y'].isin(df1['Y']))], how='inner', on=['X', 'Y'])

result = g(f1.copy(),df2.copy())
print(result)



In [ ]:

import pandas as pd


df1 = pd.DataFrame({'X': [1600, 1600, 1600, 1600, 1600, 1601, 1601, 1601, 1601],
                   'Y': ['A', 'A', 'B', 'B', 'C', 'A', 'A', 'B', 'B']})

df2 = pd.DataFrame({'X': [1600, 1600, 1600, 1600, 1601, 1601, 1601, 1601, 1601],
                   'Y': ['A', 'B', 'A', 'B', 'A', 'B', 'B', 'B', 'B']})
def g(df1, df2):
    return pd.concat([df1, df2[df2['X'].isin(df1['X'])]], axis=1)

result = g(df1.copy(),df2.copy())
print(result)



In [ ]:
teste = df2[df2['X'].isin(df1['X'])]

In [ ]:
print(teste)

In [ ]:

import pandas as pd

# Assuming df1 and df2 are your dataframes
df1 = pd.DataFrame({'X': [1600, 1600, 1600, 1600, 1600, 1601, 1601, 1601, 1601],
                    'Y': ['A', 'A', 'A', 'B', 'B', 'A', 'A', 'B', 'C']})

df2 = pd.DataFrame({'X': [1600, 1600, 1600, 1600, 1601, 1601, 1601, 1601, 1601],
                    'Y': ['A', 'A', 'B', 'B', 'A', 'A', 'B', 'B', 'B']})

# Merge the dataframes based on the specified conditions
merged_df = pd.merge(df1, df2, on=['X', 'Y'])

# Display the result
print(merged_df)


In [ ]:

import pandas as pd

# Create the first dataframe
df1 = pd.DataFrame({
    'X': [1600, 1600, 1600, 1600, 1600, 1601, 1601, 1601, 1601],
    'Y': ['A', 'A', 'A', 'B', 'B', 'A', 'A', 'B', 'C']
})

# Create the second dataframe
df2 = pd.DataFrame({
    'X': [1600, 1600, 1600, 1600, 1601, 1601, 1601, 1601, 1601],
    'Y': ['A', 'A', 'B', 'B', 'A', 'A', 'B', 'B', 'B']
})

# Merge dataframes based on specified conditions
merged_df = pd.merge(df1, df2, on=['X', 'Y'])

# Concatenate dataframes horizontally
result_df = pd.concat([df1[merged_df.index], df2[merged_df.index]], axis=1)

print(result_df)


In [ ]:
import pandas as pd

# Create sample dataframes
df1 = pd.DataFrame({'X': [1600, 1600, 1600, 1601, 1601, 1601], 'Y': ['A', 'A', 'B', 'A', 'A', 'C']})
df2 = pd.DataFrame({'X': [1600, 1600, 1600, 1601, 1601,1601,1601], 'Y': ['A', 'B', 'B', 'A', 'B', 'B', 'B']})

# Function to check if values in one dataframe exist in the same row of the other
def check_matching_values(row1, row2):
    return row1['X'] in row2['X'] and row1['Y'] in row2['Y'] and row2['X'] in row1['X'] and row2['Y'] in row1['Y']

# Concatenate dataframes using a custom approach based on conditions
result = pd.DataFrame()
for i in range(df1.shape[0]):
    for j in range(df2.shape[0]):
        if check_matching_values(df1.iloc[i], df2.iloc[j]):
            result = pd.concat([result, pd.DataFrame([df1.iloc[i], df2.iloc[j]])], ignore_index=True)

print(result)


In [ ]:
import pandas as pd

# Sample DataFrames (replace with your actual data)
df1 = pd.DataFrame({'X': [1600, 1600, 1600, 1601, 1601, 1601, 1601],
                   'Y': ['A', 'A', 'B', 'A', 'B', 'A', 'C']})
df2 = pd.DataFrame({'X': [1600, 1600, 1600, 1601, 1601, 1601, 1601],
                   'Y': ['A', 'B', 'A', 'A', 'B', 'B', 'B']})

# Function to check if values in a row of df1 exist in df2
def check_row_values(row, df2):
    return (row[0] in df2['X']) & (row[1] in df2['Y'])

# Create a new DataFrame to store concatenated rows
df_combined = pd.DataFrame()

# Iterate over rows of df1 and check for matching rows in df2
for index, row in df1.iterrows():
    if check_row_values(row, df2):
        # If matching row found, concatenate
        df_combined = pd.concat([df_combined, row.to_frame().T], ignore_index=True)
        print(df_combined)
       
# Add matching rows from df2 (if not already added)
for index, row in df2.iterrows():
    if not (row[0] in df_combined['X'] and row[1] in df_combined['Y']):
        df_combined = pd.concat([df_combined, row.to_frame().T], ignore_index=True)

# Print the concatenated DataFrame
print(df_combined)


In [ ]:
import pandas as pd

# Create the sample dataframes
df1 = pd.DataFrame({'X': [1600, 1600, 1600, 1601, 1601, 1601],
                   'Y': ['A', 'A', 'A', 'B', 'B', 'C']})
df2 = pd.DataFrame({'X': [1600, 1600, 1600, 1601, 1601, 1601],
                   'Y': ['A', 'A', 'B', 'B', 'A', 'B']})

# Function to compare values and concatenate if matched
def compare_and_concate(df1, df2):
    # Ensure consistent index length (if necessary)
    df1, df2 = df1.reindex(range(max(len(df1), len(df2))), fill_value=pd.NA), \
               df2.reindex(range(max(len(df1), len(df2))), fill_value=pd.NA)

    # Create a temporary column for comparisons
    df1['Match'] = df1['X'] == df2['X']
    df2['Match'] = df1['Match']

    # Concatenate matching rows
    concatenated_df = pd.concat([df1[df1['Match']], df2[df2['Match']]], axis=1)

    # Drop temporary column
    concatenated_df.drop('Match', axis=1, inplace=True)

    return concatenated_df

# Concatenate the dataframes
result = compare_and_concate(df1.copy(), df2.copy())

# Print the concatenated dataframe
print(result)


In [ ]:
import pandas as pd

# Create example dataframes
df1 = pd.DataFrame({'X': [1600, 1600, 1600, 1600, 1601, 1601, 1601, 1601],
                   'Y': ['A', 'A', 'A', 'B', 'A', 'A', 'C', 'B']})
df2 = pd.DataFrame({'X': [1600, 1600, 1600, 1601, 1601, 1601, 1601],
                   'Y': ['A', 'A', 'B', 'A', 'B', 'B', 'B']})

# Function to concatenate dataframes based on matching conditions
def conditional_concat(df1, df2):
    """Concatenates two DataFrames horizontally based on column values.

    Args:
        df1 (pd.DataFrame): The first DataFrame.
        df2 (pd.DataFrame): The second DataFrame.

    Returns:
        pd.DataFrame: The concatenated DataFrame.
    """

    # Create an empty list to store matching rows
    matched_rows = []

    # Iterate through rows in df1
    for i in df1.index:
        # Check if corresponding row in df2 exists with matching X and Y values
        if any((df2['X'] == df1.loc[i, 'X']) & (df2['Y'] == df1.loc[i, 'Y'])):
            # Add matching rows from both DataFrames
            matched_rows.append(pd.concat([df1.loc[i], df2.loc[df2['X'] == df1.loc[i, 'X'], 'Y']], axis=1))

    return pd.concat(matched_rows)

# Concatenate the DataFrames
result = conditional_concat(df1.copy(), df2.copy())

# Print the result
print(result)


In [ ]:
import pandas as pd

# Create sample DataFrames
df1 = pd.DataFrame({'X': [1600, 1600, 1600, 1601, 1601, 1601, 1601],
                   'Y': ['A', 'A', 'A', 'B', 'B', 'A', 'C']})
df2 = pd.DataFrame({'X': [1600, 1600, 1600, 1601, 1601],
                   'Y': ['A', 'A', 'B', 'B', 'A']})

# Function to concatenate DataFrames based on conditions
def conditional_concat(df1, df2):
    # Iterate through rows of df1
    result = pd.DataFrame()
    for _, row1 in df1.iterrows():
        # Find corresponding row in df2 with matching X and Y
        matching_row2 = df2[(df2['X'] == row1['X']) & (df2['Y'] == row1['Y'])]

        # Concatenate if found, otherwise skip
        if not matching_row2.empty:
            result = pd.concat([result, pd.concat([row1, matching_row2.iloc[0]])], ignore_index=True)
    return result

# Concatenate DataFrames using the function
result = conditional_concat(df1, df2)

# Print the concatenated DataFrame
print(result)


In [ ]:
import pandas as pd

# Create sample DataFrames (replace with your actual data)
df1 = pd.DataFrame({'X': [1600, 1600, 1600, 1601, 1601, 1601, 1601],
                   'Y': ['A', 'A', 'A', 'B', 'B', 'A', 'C']})
df2 = pd.DataFrame({'X': [1600, 1600, 1600, 1601, 1601],
                   'Y': ['A', 'A', 'B', 'B', 'A']})

# Function to concatenate DataFrames based on conditions
def conditional_concat(df1, df2):
    result = pd.DataFrame()
    group = pd.DataFrame()
    i1, i2 = 0, 0  # Initialize indices for df1 and df2
    while True:
        # Check if both DataFrames are exhausted
        if i1 >= df1.shape[0] or i2 >= df2.shape[0]:
            if not group.empty:
                result = pd.concat([result, group], ignore_index=True)
            break

        # Get current rows from both DataFrames
        row1 = df1.iloc[i1]
        row2 = df2.iloc[i2]

        # Check if values match and update indices
        if row1['X'] == row2['X'] and row1['Y'] == row2['Y']:
            group = pd.concat([group, pd.concat([row1, row2])])
            i1 += 1
            i2 += 1
        else:
            # Different values encountered, concatenate group and start a new one
            if not group.empty:
                result = pd.concat([result, group], ignore_index=True)
            group = pd.DataFrame()

            # Find next matching indices in df1 and df2
            while i1 < df1.shape[0] and (row1['X'] != df2.iloc[i2]['X'] or row1['Y'] != df2.iloc[i2]['Y']):
                i1 += 1
            while i2 < df2.shape[0] and (row2['X'] != df1.iloc[i1]['X'] or row2['Y'] != df1.iloc[i1]['Y']):
                i2 += 1

    return result

# Concatenate DataFrames using the function
result = conditional_concat(df1, df2)

# Print the concatenated DataFrame
print(result)


In [ ]:
import pandas as pd

df1 = pd.DataFrame({'X': [1600, 1600, 1600, 1600, 1600, 1601, 1601, 1601, 1601],
'Y': ['A', 'A', 'B', 'B', 'C', 'A', 'A', 'B', 'B']})

df2 = pd.DataFrame({'X': [1600, 1600, 1600, 1600, 1601, 1601, 1601, 1601, 1601],
'Y': ['A', 'A', 'A', 'B', 'A', 'B', 'B', 'B', 'B']})

import pandas as pd

def g(df1, df2):
    # Create a boolean mask to select rows from df2 where X and Y match df1's first row
    mask = (df2['X'] == df1.iloc[0]['X']) & (df2['Y'] == df1.iloc[0]['Y'])

    # Concatenate starting from the first row of df1, then include rows from df2 that match
    return pd.concat([df1, df2.loc[mask][df2.index >= df1.index.min()]])

result = g(df1.copy(), df2.copy())
print(result)




In [ ]:
import pandas as pd

def equalize_dataframe_sizes(df1, df2):
    """Equalizes the sizes of two DataFrames by dropping rows from the larger one.

    Args:
        df1 (pd.DataFrame): The first DataFrame.
        df2 (pd.DataFrame): The second DataFrame.

    Returns:
        tuple: A tuple containing the equalized DataFrames (df1, df2).

    Raises:
        ValueError: If both DataFrames have the same size.
    """

    if df1.shape[0] == df2.shape[0]:
        raise ValueError("Both DataFrames already have the same size.")

    larger_df, smaller_df = (df1, df2) if df1.shape[0] > df2.shape[0] else (df2, df1)

    # Drop rows from the larger DataFrame to match the smaller one
    larger_df = larger_df.iloc[: smaller_df.shape[0], :]

    return larger_df, smaller_df

# Example usage
df1 = pd.DataFrame({'A': [1, 2, 3, 4, 5], 'B': ['a', 'b', 'c', 'd', 'e']})
df2 = pd.DataFrame({'A': [10, 20, 30], 'B': ['x', 'y', 'z']})

try:
    df1_equal, df2_equal = equalize_dataframe_sizes(df1.copy(), df2.copy())
    print("Equalized DataFrames:")
    print(df1_equal)
    print(df2_equal)
except ValueError as e:
    print(e)
